# Word guesser: model benchmark on Kaggle

Before running:
1. In the right-hand settings panel, set **Accelerator** to a GPU (T4 x2 is best) and turn **Internet** on.
   If either is greyed out, Kaggle probably wants phone verification in your account settings.
2. Run the cells **top to bottom**, one at a time. The installs and model downloads take a few minutes.
3. Copy the output of the last cell back into the chat.

Everything installed here disappears when the session ends, so a new session means re-running from the top.


In [ ]:
!nvidia-smi

In [ ]:
# Install Ollama and its Python library
!apt-get install -y -qq lshw pciutils zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q -U ollama


In [ ]:
# Start the Ollama server in the background
import os, subprocess, time

env = dict(os.environ, OLLAMA_KEEP_ALIVE="-1")
server = subprocess.Popen(["ollama", "serve"], env=env,
                          stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)
print(subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout)


In [ ]:
%%writefile guesser.py
"""
20Q-style word guesser for an Arduino Uno + I2C LCD.

The laptop does the thinking, the Uno just shows text.

How it works (same ideas as the 2003 Radica 20Q toy):
  * 5 answer levels: yes / probably / sometimes / probably not / no
  * every word keeps a SCORE instead of being deleted on a bad answer,
    so one sloppy answer doesn't ruin the game
  * it asks the question that best splits the likely words
  * it keeps asking and guessing again after a wrong guess
  * it LEARNS: new words and corrected answers are saved to words.json

Run:     python guesser.py
Grow it: python guesser.py prompt newwords.txt   (writes fill_prompt.txt)
         python guesser.py merge fill_result.json
"""
import json
import math
import os
import sys
import time

try:
    import serial
except ImportError:
    serial = None

# ---------------------------------------------------------------- settings
PORT = "COM13"
BAUD = 9600
HERE = os.path.dirname(os.path.abspath(__file__))
DATA_FILE = os.path.join(HERE, "words.json")

MIN_Q = 5            # earliest a confident guess is allowed
CONFIDENT = 0.60     # guess early when the top word has this probability
GUESS_AT = (20, 23, 26, 29)   # forced guesses, like 20Q
MAX_Q = 30
CANDIDATES = 250     # only the likeliest words are used to choose questions
SEED_CONF = 3        # how strongly the built-in answers resist being changed

LEVELS = (1.0, 0.75, 0.5, 0.25, 0.0)
ANSWERS = {"1": 1.0, "y": 1.0, "2": 0.75, "p": 0.75, "3": 0.5, "s": 0.5,
           "d": 0.5, "4": 0.25, "m": 0.25, "5": 0.0, "n": 0.0}
LEGEND = "  1=yes  2=probably  3=sometimes/unsure  4=probably not  5=no  (q=quit)"

# ----------------------------------------------------------- built-in data
# Questions must fit on a 16x2 LCD (32 characters max).
QUESTIONS = {
    "alive": "Is it alive?",
    "animal": "Is it an animal?",
    "plant": "Is it a plant?",
    "manmade": "Is it man-made?",
    "hand": "Fits in your hand?",
    "bigcar": "Bigger than a car?",
    "bigperson": "Bigger than a person?",
    "electric": "Does it use electricity?",
    "edible": "Can you eat it?",
    "sweet": "Does it taste sweet?",
    "home": "Found in most homes?",
    "kitchen": "Found in a kitchen?",
    "outdoors": "Found outdoors?",
    "wheels": "Does it have wheels?",
    "noise": "Does it make noise?",
    "moves": "Can it move by itself?",
    "flies": "Can it fly?",
    "water": "Is it found in water?",
    "metal": "Is it made of metal?",
    "wood": "Is it made of wood?",
    "fabric": "Is it made of fabric?",
    "glass": "Is it made of glass?",
    "plastic": "Is it made of plastic?",
    "wear": "Do you wear it?",
    "tool": "Is it a tool?",
    "toy": "Is it a toy or game?",
    "pet": "Is it a common pet?",
    "fur": "Does it have fur or hair?",
    "legs": "Does it have legs?",
    "hot": "Can it get hot?",
    "cold": "Is it often cold?",
    "sit": "Can you sit on it?",
    "read": "Do you read it?",
    "screen": "Does it have a screen?",
    "sharp": "Is it sharp?",
    "round": "Is it round?",
    "school": "Used at school?",
    "sport": "Used in sports?",
    "music": "Used for music?",
    "danger": "Can it hurt you?",
    "sky": "Is it in the sky?",
    "liquid": "Is it a liquid?",
    "soft": "Is it soft?",
    "vehicle": "Is it a vehicle?",
}

# word -> tags that are YES.  "~tag" = sometimes.  Anything not listed = no.
SEED = {
    "dog": "alive animal fur legs moves noise pet outdoors home soft ~danger",
    "cat": "alive animal fur legs moves noise pet home outdoors soft",
    "horse": "alive animal fur legs moves noise outdoors bigperson ~danger ~sport",
    "cow": "alive animal fur legs moves noise outdoors bigperson",
    "elephant": "alive animal legs moves noise outdoors bigperson ~bigcar danger",
    "fish": "alive animal water moves ~edible ~pet ~hand",
    "bird": "alive animal flies legs moves noise outdoors sky hand ~pet",
    "snake": "alive animal moves outdoors danger ~pet ~water",
    "shark": "alive animal water moves danger bigperson",
    "spider": "alive animal legs moves outdoors hand ~danger",
    "frog": "alive animal legs moves water outdoors hand ~noise ~pet",
    "tree": "alive plant outdoors wood bigperson ~edible",
    "flower": "alive plant outdoors soft hand ~home",
    "grass": "alive plant outdoors soft",
    "apple": "~plant edible sweet hand round kitchen home",
    "banana": "~plant edible sweet hand kitchen home soft",
    "pizza": "manmade edible kitchen home hot ~round",
    "bread": "manmade edible kitchen home soft hand ~sweet",
    "cheese": "manmade edible kitchen home soft hand",
    "egg": "edible kitchen home hand round ~hot",
    "milk": "manmade edible liquid kitchen home ~cold ~sweet",
    "water": "~edible liquid home kitchen outdoors water ~cold ~hot",
    "ice cream": "manmade edible sweet cold kitchen home soft ~hand",
    "chocolate": "manmade edible sweet hand home kitchen ~soft",
    "cookie": "manmade edible sweet hand kitchen home round",
    "pencil": "manmade hand home school wood sharp",
    "pen": "manmade hand home school plastic ~metal",
    "book": "manmade hand home school read",
    "paper": "manmade hand home school read",
    "scissors": "manmade hand home school metal tool sharp ~plastic",
    "knife": "manmade hand kitchen home metal tool sharp danger",
    "spoon": "manmade hand kitchen home metal tool ~plastic",
    "fork": "manmade hand kitchen home metal tool sharp ~plastic",
    "cup": "manmade hand kitchen home ~glass ~plastic",
    "plate": "manmade kitchen home round ~hand ~glass",
    "bottle": "manmade hand kitchen home ~glass ~plastic",
    "key": "manmade hand home metal ~tool",
    "phone": "manmade hand home electric screen noise glass plastic metal",
    "laptop": "manmade electric screen home school plastic metal hot ~noise",
    "computer": "manmade electric screen home school plastic metal hot ~noise",
    "television": "manmade electric screen home noise glass plastic hot",
    "clock": "manmade home noise round ~electric",
    "lamp": "manmade home electric glass metal hot ~plastic",
    "fan": "manmade home electric noise plastic metal ~moves",
    "refrigerator": "manmade home kitchen electric noise cold metal ~bigperson",
    "chair": "manmade home sit wood school ~metal ~plastic ~fabric",
    "couch": "manmade home sit soft fabric ~bigperson ~wood",
    "bed": "manmade home sit soft fabric ~bigperson ~wood ~metal",
    "table": "manmade home kitchen school wood ~metal ~glass",
    "door": "manmade home wood bigperson ~metal ~glass",
    "window": "manmade home glass",
    "mirror": "manmade home glass ~hand",
    "shoe": "manmade wear home fabric ~outdoors ~hand ~soft",
    "shirt": "manmade wear home fabric soft",
    "hat": "manmade wear home fabric soft ~outdoors",
    "glasses": "manmade wear hand glass plastic home",
    "watch": "manmade wear hand metal electric ~screen ~noise round ~glass",
    "backpack": "manmade ~wear school fabric home ~outdoors soft",
    "umbrella": "manmade ~hand outdoors fabric metal home ~tool",
    "ball": "manmade toy sport round ~hand outdoors ~plastic",
    "bicycle": "manmade wheels vehicle outdoors metal sport",
    "car": "manmade wheels vehicle outdoors metal moves electric noise bigperson",
    "bus": "manmade wheels vehicle outdoors metal moves noise bigperson bigcar",
    "train": "manmade wheels vehicle outdoors metal moves noise bigperson bigcar",
    "airplane": "manmade vehicle flies sky metal moves noise bigperson bigcar",
    "rocket": "manmade vehicle flies sky metal moves noise bigperson bigcar hot danger",
    "helicopter": "manmade vehicle flies sky metal moves noise bigperson ~bigcar",
    "boat": "manmade vehicle water moves ~noise bigperson ~bigcar ~wood ~metal",
    "guitar": "manmade music wood noise home",
    "piano": "manmade music wood home noise bigperson ~metal",
    "drum": "manmade music noise round ~wood",
    "hammer": "manmade hand tool metal wood home ~danger",
    "screwdriver": "manmade hand tool metal home ~sharp",
    "saw": "manmade hand tool metal sharp danger ~wood",
    "battery": "manmade hand electric metal ~home",
    "light bulb": "manmade hand electric glass home hot round",
    "robot": "manmade electric moves metal plastic ~noise ~toy ~screen ~legs ~wheels",
    "sun": "sky hot round outdoors bigcar bigperson danger",
    "moon": "sky round outdoors bigcar bigperson cold",
    "star": "sky hot round outdoors bigcar bigperson",
    "cloud": "sky outdoors ~water soft bigcar bigperson ~cold",
    "rain": "water liquid outdoors sky ~cold ~noise",
    "snow": "water outdoors cold ~sky soft",
    "rock": "outdoors ~hand ~bigperson",
    "mountain": "outdoors bigcar bigperson",
    "river": "water liquid outdoors bigcar bigperson ~noise ~moves",
    "ocean": "water liquid outdoors bigcar bigperson ~noise ~moves",
    "fire": "hot ~outdoors danger ~noise ~moves ~home",
    "sand": "outdoors ~water ~soft ~hand",
    "toothbrush": "manmade hand home plastic ~soft",
    "soap": "manmade hand home ~liquid",
    "towel": "manmade home fabric soft",
    "pillow": "manmade home soft fabric",
    "blanket": "manmade home soft fabric ~wear",
    "coin": "manmade hand metal round ~home",
    "candle": "manmade hand home hot ~danger",
    "camera": "manmade hand electric screen glass plastic ~noise home",
    "headphones": "manmade electric wear music noise plastic home ~hand",
}


def build_seed():
    words = {}
    for name, tags in SEED.items():
        vals = {q: 0.0 for q in QUESTIONS}
        for tag in tags.split():
            if tag.startswith("~"):
                vals[tag[1:]] = 0.5
            else:
                vals[tag] = 1.0
        words[name] = {"v": vals, "n": {q: SEED_CONF for q in QUESTIONS},
                       "plays": 1}
    return {"questions": dict(QUESTIONS), "words": words, "games": 0, "wins": 0}


# ------------------------------------------------------------- save / load
def load():
    if os.path.exists(DATA_FILE):
        with open(DATA_FILE, encoding="utf-8") as f:
            return json.load(f)
    return build_seed()


def save(db):
    tmp = DATA_FILE + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(db, f)
    os.replace(tmp, DATA_FILE)


# ----------------------------------------------------------------- display
def split_lines(text):
    if len(text) <= 16:
        return text, ""
    cut = text.rfind(" ", 0, 17)
    cut = cut if cut > 0 else 16
    return text[:cut][:16], text[cut:].strip()[:16]


class Display:
    """Sends text to the Uno. If the Uno isn't there, it still prints."""

    def __init__(self):
        self.ser = None
        if serial is None:
            print("(pyserial not installed - running without the LCD)")
            return
        try:
            self.ser = serial.Serial(PORT, BAUD, timeout=1)
            time.sleep(2)  # the Uno resets when the port opens
        except Exception as err:
            print(f"(could not open {PORT}: {err})")
            print("(running without the LCD - close the Serial Monitor?)")

    def show(self, text):
        top, bottom = split_lines(text)
        print(f"  [LCD] {top} / {bottom}")
        if self.ser:
            self.ser.write(f"{top}|{bottom}\n".encode())

    def close(self):
        if self.ser:
            self.ser.close()


# ------------------------------------------------------------------- input
def ask_answer(text):
    """Return an answer level 0..1 for a question."""
    while True:
        a = input(f"{text}\n{LEGEND}\n> ").strip().lower()
        if a == "q":
            raise KeyboardInterrupt
        if a in ANSWERS:
            return ANSWERS[a]


def yes_no(text):
    while True:
        a = input(text).strip().lower()
        if a == "q":
            raise KeyboardInterrupt
        if a in ("y", "yes", "1"):
            return True
        if a in ("n", "no", "5"):
            return False


def read_line(text):
    return input(text).strip()


# ----------------------------------------------------------------- the AI
def val(word, q):
    return word["v"].get(q, 0.5)      # unknown = 0.5


def lik(a, v):
    """How well does answer a fit a word whose stored value is v?"""
    return 0.05 + 0.95 * (1 - abs(a - v))


def posterior(logp):
    m = max(logp.values())
    e = {w: math.exp(s - m) for w, s in logp.items()}
    z = sum(e.values())
    return {w: x / z for w, x in e.items()}


def entropy(ps):
    return -sum(p * math.log(p) for p in ps if p > 0)


def pick_question(db, post, asked):
    """Choose the question that is expected to narrow things down the most."""
    ranked = sorted(post.items(), key=lambda kv: -kv[1])[:CANDIDATES]
    z = sum(p for _, p in ranked)
    cands = [(db["words"][w], p / z) for w, p in ranked]
    best_q = None
    best_h = entropy([p for _, p in cands]) - 0.01   # must help at least a bit
    for q in db["questions"]:
        if q in asked:
            continue
        rows = []
        tot = [0.0] * 5
        for word, p in cands:
            v = val(word, q)
            ls = [lik(a, v) for a in LEVELS]
            s = sum(ls)
            row = [p * x / s for x in ls]
            rows.append(row)
            for i, x in enumerate(row):
                tot[i] += x
        h = 0.0
        for i in range(5):
            if tot[i] > 0:
                h += tot[i] * entropy([r[i] / tot[i] for r in rows])
        if h < best_h:
            best_q, best_h = q, h
    return best_q


def play(db, lcd):
    """One game. Returns (guessed_word_or_None, answers_given, won)."""
    words = db["words"]
    logp = {w: math.log(1 + d["plays"]) for w, d in words.items()}
    asked = {}
    rejected = set()
    last_guess_n = -1

    lcd.show("Think of a word!")
    read_line("Press Enter when ready...")

    while True:
        n = len(asked)
        post = posterior(logp)
        top, p_top = max(post.items(), key=lambda kv: kv[1])
        want_guess = n != last_guess_n and (
            (n >= MIN_Q and p_top >= CONFIDENT) or n in GUESS_AT)

        q = None
        if not want_guess and n < MAX_Q:
            q = pick_question(db, post, asked)
        if q is not None:
            text = db["questions"][q]
            lcd.show(text)
            a = ask_answer(f"Q{n + 1}: {text}")
            asked[q] = a
            for w, d in words.items():
                logp[w] += math.log(lik(a, val(d, q)))
            continue

        # time to guess, or nothing useful left to ask
        if n == last_guess_n or top in rejected:
            lcd.show("You got me!")
            return None, asked, False
        last_guess_n = n
        lcd.show(f"Is it {top}?")
        if yes_no(f"Is it {top}? (y/n) > "):
            lcd.show("I win!")
            return top, asked, True
        rejected.add(top)
        logp[top] -= 8


def learn(db, name, asked):
    """Fold a finished game into what the word 'looks like'."""
    w = db["words"].setdefault(name, {"v": {}, "n": {}, "plays": 0})
    w["plays"] += 1
    for q, a in asked.items():
        n = w["n"].get(q, 0)
        old = w["v"].get(q, a)
        w["v"][q] = old + (a - old) / (n + 1)
        w["n"][q] = n + 1


# --------------------------------------------- growing the word list (LLM)
def make_prompt(path):
    db = load()
    with open(path, encoding="utf-8") as f:
        new = [line.strip().lower() for line in f if line.strip()]
    new = [w for w in dict.fromkeys(new) if w not in db["words"]]
    qs = list(db["questions"].values())
    lines = [
        "For each word below, answer every question about the thing the word",
        "names. Use one digit per question: 4=yes, 3=probably, 2=sometimes or",
        "depends, 1=probably not, 0=no.",
        f"Reply with ONLY a JSON object mapping each word to a string of exactly",
        f"{len(qs)} digits, one per question, in the numbered order. No other text.",
        "", "Questions:",
    ]
    lines += [f"{i + 1}. {t}" for i, t in enumerate(qs)]
    lines += ["", "Words:"] + new
    out = os.path.join(HERE, "fill_prompt.txt")
    with open(out, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"Wrote {out} with {len(new)} new words.")
    print("Paste it into a Claude chat (30-50 words per paste works best),")
    print("save the JSON reply as fill_result.json, then run:")
    print("  python guesser.py merge fill_result.json")


def merge(path):
    db = load()
    with open(path, encoding="utf-8") as f:
        text = f.read().strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0]
    data = json.loads(text)
    qids = list(db["questions"])
    added = 0
    for word, digits in data.items():
        digits = str(digits).strip()
        if len(digits) != len(qids) or any(c not in "01234" for c in digits):
            print(f"skipped '{word}': need exactly {len(qids)} digits 0-4")
            continue
        db["words"][word.strip().lower()] = {
            "v": {q: int(c) / 4 for q, c in zip(qids, digits)},
            "n": {q: 2 for q in qids},
            "plays": 1,
        }
        added += 1
    save(db)
    print(f"Added {added} words. The list now has {len(db['words'])}.")


# -------------------------------------------------------------------- main
def main():
    args = sys.argv[1:]
    if len(args) == 2 and args[0] == "prompt":
        return make_prompt(args[1])
    if len(args) == 2 and args[0] == "merge":
        return merge(args[1])

    db = load()
    lcd = Display()
    print(f"I know {len(db['words'])} words. Think of an object, animal or food.")
    try:
        while True:
            guess, asked, won = play(db, lcd)
            db["games"] += 1
            if won:
                db["wins"] += 1
                learn(db, guess, asked)
            else:
                name = read_line("What were you thinking of? > ").lower()
                if name:
                    learn(db, name, asked)
                    lcd.show("Learned it!")
            save(db)
            if not yes_no("Play again? (y/n) > "):
                break
    except (KeyboardInterrupt, EOFError):
        print()
    finally:
        save(db)
        if db["games"]:
            print(f"Record: {db['wins']} wins in {db['games']} games.")
        lcd.close()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile benchmark_models.py
"""
Compare local Ollama models on the guesser's hand-written words.

For each model it asks about N sample words (all questions in one call),
then reports how often the model agrees with the hand-written answers
and how long each word takes.

Run it from the same folder as guesser.py:
    python benchmark_models.py
"""
import json
import random
import time

import ollama

from guesser import build_seed

# ---------------------------------------------------------------- settings
MODELS = ["qwen3.5:9b", "gemma4:12b"]   # compared on the same 12 words
N_WORDS = 12                            # more words = slower but fairer
SEED = 7                                # same sample for every model
SHOW = {"airplane", "egg", "clock"}     # print the disagreements for these

LABELS = ["yes", "probably", "sometimes", "probably not", "no"]
VALUE = {"yes": 1.0, "probably": 0.75, "sometimes": 0.5,
         "probably not": 0.25, "no": 0.0}

db = build_seed()
QUESTIONS = db["questions"]

schema = {
    "type": "object",
    "properties": {q: {"type": "string", "enum": LABELS} for q in QUESTIONS},
    "required": list(QUESTIONS),
}

# Fixed text first, the word last: Ollama can reuse the repeated beginning.
HEADER = (
    "You help build a guessing game. For the thing named by the word at the "
    "end, answer every question about a typical example of it using exactly "
    "one of: yes, probably, sometimes, probably not, no.\n\nQuestions:\n"
    + "\n".join(f"{q}: {t}" for q, t in QUESTIONS.items())
    + "\n\nWord: "
)


def score_word(model, word):
    reply = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": HEADER + word}],
        format=schema,
        think=False,
        options={"temperature": 0, "num_ctx": 2048},
    )
    raw = json.loads(reply["message"]["content"])
    return {q: VALUE[raw[q]] for q in QUESTIONS if raw.get(q) in VALUE}


def compare(truth, got):
    """Count agreement on the clear-cut answers (hand-written 0 or 1)."""
    right = total = yes_right = yes_total = false_yes = 0
    err = 0.0
    for q, t in truth["v"].items():
        if q not in got:
            continue
        err += abs(got[q] - t)
        if t in (0.0, 1.0):          # skip the "sometimes" answers
            total += 1
            right += (got[q] > 0.5) == (t == 1.0)
            if t == 1.0:             # most answers are "no", so track "yes" alone
                yes_total += 1
                yes_right += got[q] > 0.5
            else:
                false_yes += got[q] > 0.5
    return right, total, yes_right, yes_total, false_yes, err / len(QUESTIONS)


def main():
    random.seed(SEED)
    sample = random.sample(list(db["words"]), N_WORDS)
    print(f"Testing on: {', '.join(sample)}\n")

    for model in MODELS:
        print(f"=== {model} ===")
        try:
            score_word(model, "warmup")      # loads the model, not timed
        except Exception as err:
            print(f"  skipped ({err})\n")
            continue
        right = total = yes_right = yes_total = false_yes = 0
        errs = []
        start = time.time()
        for word in sample:
            try:
                got = score_word(model, word)
            except Exception as err:
                print(f"  {word}: failed ({err})")
                continue
            r, t, yr, yt, fy, e = compare(db["words"][word], got)
            right, total = right + r, total + t
            yes_right, yes_total = yes_right + yr, yes_total + yt
            false_yes += fy
            errs.append(e)
            print(f"  {word:<12} {r}/{t} match, {yr}/{yt} yeses found, "
                  f"{fy} false yeses")
            if word in SHOW:
                for q, tv in db["words"][word]["v"].items():
                    if tv in (0.0, 1.0) and q in got and \
                            (got[q] > 0.5) != (tv == 1.0):
                        mine = "yes" if tv else "no"
                        print(f"      {QUESTIONS[q]:<28} mine={mine:<3} "
                              f"model={got[q]}")
        secs = (time.time() - start) / max(len(errs), 1)
        if total:
            print(f"  agreement: {right / total:.0%}   "
                  f"(a model that always says no scores "
                  f"{1 - yes_total / total:.0%})")
            print(f"  yeses found: {yes_right / max(yes_total, 1):.0%}   "
                  f"false yeses: {false_yes}   "
                  f"avg error: {sum(errs) / len(errs):.2f}   "
                  f"{secs:.1f} s/word")
            print(f"  rough time for 2,000 words: {secs * 2000 / 3600:.1f} hours\n")


if __name__ == "__main__":
    main()


In [ ]:
# Download the models (about 6 GB and 8 GB)
import subprocess

for m in ["qwen3.5:9b", "gemma4:12b"]:
    r = subprocess.run(["ollama", "pull", m], capture_output=True, text=True)
    print(m, "OK" if r.returncode == 0 else "FAILED: " + r.stderr[-300:])


In [ ]:
!python benchmark_models.py

In [ ]:
!ollama ps